# Crawl Nghị định 168/2024/NĐ-CP - Xử phạt vi phạm giao thông đường bộ

Notebook này trích xuất nội dung từ file PDF **Nghị định 168/2024/NĐ-CP** bao gồm:
- Quy định xử phạt vi phạm hành chính về trật tự, an toàn giao thông đường bộ
- Trừ điểm, phục hồi điểm giấy phép lái xe
- Phân tích cấu trúc: Chương, Mục, Điều với mức phạt cụ thể
- Lưu dữ liệu JSON phục vụ **Retrieval chatbot AI** trả lời câu hỏi về mức xử phạt

## 1. Cài đặt và Import thư viện

In [1]:
# Cài đặt thư viện cần thiết
import sys
!{sys.executable} -m pip install -q pdfplumber

'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
# Import các thư viện cần thiết
import json
import re
import os
import io
from pathlib import Path
from datetime import datetime

import pdfplumber

In [3]:
# Cấu hình đường dẫn
BASE_DIR = Path(r"C:\Users\ASUS-PRO\Desktop\Traffic_Law_Chatbot")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PDF_PATH = RAW_DIR / "ND-168-PDF.pdf"
OUTPUT_JSON = RAW_DIR / "nd168_xu_phat_2024.json"
OUTPUT_PROCESSED = PROCESSED_DIR / "nd168_penalties.json"

print(f"PDF: {PDF_PATH}")
print(f"Tồn tại: {PDF_PATH.exists()}")
print(f"Kích thước: {PDF_PATH.stat().st_size / 1024:.1f} KB")

PDF: C:\Users\ASUS-PRO\Desktop\Traffic_Law_Chatbot\data\raw\ND-168-PDF.pdf
Tồn tại: True
Kích thước: 838.0 KB


## 2. Trích xuất văn bản từ PDF

In [4]:
# Trích xuất toàn bộ văn bản từ PDF
print("Đang trích xuất văn bản từ Nghị định 168/2024/NĐ-CP...")

all_text = ""
pages_data = []

with pdfplumber.open(str(PDF_PATH)) as pdf:
    total_pages = len(pdf.pages)
    print(f"Tổng số trang: {total_pages}")
    
    for i, page in enumerate(pdf.pages):
        page_text = page.extract_text()
        if page_text:
            pages_data.append({
                "page_number": i + 1,
                "content": page_text
            })
            all_text += page_text + "\n\n"
        
        if (i + 1) % 20 == 0:
            print(f"  Đã xử lý {i + 1}/{total_pages} trang...")

print(f"\n✓ Trích xuất thành công {len(pages_data)} trang")
print(f"✓ Tổng độ dài văn bản: {len(all_text):,} ký tự")

Đang trích xuất văn bản từ Nghị định 168/2024/NĐ-CP...
Tổng số trang: 88
  Đã xử lý 20/88 trang...
  Đã xử lý 40/88 trang...
  Đã xử lý 60/88 trang...
  Đã xử lý 80/88 trang...

✓ Trích xuất thành công 88 trang
✓ Tổng độ dài văn bản: 255,063 ký tự


## 3. Phân tích cấu trúc Nghị định

Nghị định 168/2024/NĐ-CP có cấu trúc:
- **Chương I**: Quy định chung
- **Chương II**: Xử phạt vi phạm (các Điều quy định mức phạt cụ thể theo loại phương tiện)
- **Chương III**: Trừ điểm giấy phép lái xe
- **Chương IV**: Thẩm quyền xử phạt
- **Chương V**: Điều khoản thi hành

In [5]:
# Phân tích cấu trúc: Chương, Mục, Điều
print("Đang phân tích cấu trúc Nghị định 168...")

chapter_pattern = re.compile(r'Chương\s+([IVXLCDM]+)\s*[:\n]?\s*(.+?)(?=\n|$)', re.IGNORECASE)
section_pattern = re.compile(r'Mục\s+(\d+)\s*[:\.]?\s*(.+?)(?=\n|$)', re.IGNORECASE)
article_pattern = re.compile(r'Điều\s+(\d+)\s*\.?\s*(.+?)(?=\n|$)', re.IGNORECASE)

chapters = []
current_chapter = None
current_section = None
current_article = None

lines = all_text.split('\n')

for line in lines:
    line_stripped = line.strip()
    if not line_stripped:
        continue
    
    # Kiểm tra Chương
    chapter_match = chapter_pattern.search(line_stripped)
    if chapter_match:
        if current_chapter:
            chapters.append(current_chapter)
        current_chapter = {
            "chapter_number": chapter_match.group(1),
            "chapter_title": chapter_match.group(2).strip(),
            "sections": [],
            "articles": []
        }
        current_section = None
        current_article = None
        continue
    
    # Kiểm tra Mục
    section_match = section_pattern.search(line_stripped)
    if section_match and current_chapter:
        current_section = {
            "section_number": section_match.group(1),
            "section_title": section_match.group(2).strip(),
            "articles": []
        }
        current_chapter["sections"].append(current_section)
        current_article = None
        continue
    
    # Kiểm tra Điều
    article_match = article_pattern.search(line_stripped)
    if article_match:
        current_article = {
            "article_number": article_match.group(1),
            "article_title": article_match.group(2).strip(),
            "content": ""
        }
        # Thêm vào mục hoặc chương
        if current_section:
            current_section["articles"].append(current_article)
        elif current_chapter:
            current_chapter["articles"].append(current_article)
        continue
    
    # Thêm nội dung vào điều hiện tại
    if current_article:
        current_article["content"] += line_stripped + "\n"

# Thêm chương cuối cùng
if current_chapter:
    chapters.append(current_chapter)

print(f"✓ Đã phân tích {len(chapters)} chương")
for ch in chapters:
    total_arts = len(ch["articles"]) + sum(len(s["articles"]) for s in ch["sections"])
    sections_info = f", {len(ch['sections'])} mục" if ch["sections"] else ""
    print(f"  Chương {ch['chapter_number']}: {ch['chapter_title']} ({total_arts} điều{sections_info})")

Đang phân tích cấu trúc Nghị định 168...
✓ Đã phân tích 6 chương
  Chương I: I (99 điều, 5 mục)
  Chương II: I (58 điều, 2 mục)
  Chương I: I (91 điều, 1 mục)
  Chương II: Phần thứ hai của (0 điều)
  Chương III: của Nghị định này và có thẩm quyền xử phạt (1 điều)
  Chương I: V (54 điều)


## 4. Trích xuất mức phạt cụ thể

Phân tích nội dung các Điều để trích xuất thông tin mức phạt tiền, hành vi vi phạm, 
và điểm trừ trên giấy phép lái xe.

In [6]:
# Trích xuất mức phạt tiền từ nội dung các điều
print("Đang trích xuất mức phạt...")

# Pattern nhận diện mức phạt tiền (VD: "Phạt tiền từ 200.000 đồng đến 400.000 đồng")
penalty_pattern = re.compile(
    r'[Pp]hạt\s+tiền\s+từ\s+([\d.,]+)\s*đồng\s+đến\s+([\d.,]+)\s*đồng',
    re.IGNORECASE
)

# Pattern nhận diện trừ điểm GPLX
point_pattern = re.compile(
    r'[Tt]rừ\s+(\d+)\s+điểm',
    re.IGNORECASE
)

penalties = []

def get_all_articles(chapters):
    """Lấy tất cả các điều từ chương và mục"""
    articles = []
    for ch in chapters:
        for art in ch["articles"]:
            articles.append({**art, "chapter": ch["chapter_number"]})
        for sec in ch["sections"]:
            for art in sec["articles"]:
                articles.append({**art, "chapter": ch["chapter_number"], "section": sec["section_number"]})
    return articles

all_articles = get_all_articles(chapters)

for article in all_articles:
    content = article.get("content", "")
    
    # Tìm mức phạt tiền
    penalty_matches = penalty_pattern.findall(content)
    
    # Tìm điểm trừ
    point_matches = point_pattern.findall(content)
    
    if penalty_matches or point_matches:
        penalty_info = {
            "article_number": article["article_number"],
            "article_title": article["article_title"],
            "chapter": article.get("chapter", ""),
            "fine_ranges": [],
            "demerit_points": []
        }
        
        for min_fine, max_fine in penalty_matches:
            penalty_info["fine_ranges"].append({
                "min": min_fine.replace(".", "").replace(",", ""),
                "max": max_fine.replace(".", "").replace(",", ""),
                "text": f"Từ {min_fine} đồng đến {max_fine} đồng"
            })
        
        for points in point_matches:
            if points not in penalty_info["demerit_points"]:
                penalty_info["demerit_points"].append(points)
        
        penalties.append(penalty_info)

print(f"\n✓ Tìm thấy {len(penalties)} điều có quy định mức phạt")
print(f"✓ Tổng số mức phạt tiền: {sum(len(p['fine_ranges']) for p in penalties)}")
print(f"✓ Số điều có trừ điểm GPLX: {sum(1 for p in penalties if p['demerit_points'])}")

# Hiển thị mẫu
print(f"\nMẫu một số mức phạt:")
for p in penalties[:5]:
    print(f"\n  Điều {p['article_number']}: {p['article_title']}")
    for fr in p["fine_ranges"][:2]:
        print(f"    💰 {fr['text']}")
    if p["demerit_points"]:
        print(f"    📍 Trừ điểm: {', '.join(p['demerit_points'])} điểm")

Đang trích xuất mức phạt...

✓ Tìm thấy 52 điều có quy định mức phạt
✓ Tổng số mức phạt tiền: 187
✓ Số điều có trừ điểm GPLX: 0

Mẫu một số mức phạt:

  Điều 6: Xử phạt, trừ điểm giấy phép lái xe của người điều khiển xe ô tô, xe chở người bốn
    💰 Từ 400.000 đồng đến 600.000 đồng
    💰 Từ 600.000 đồng đến 800.000 đồng

  Điều 7: Xử phạt, trừ điểm giấy phép lái của người điều khiển xe mô tô, xe gắn máy, các loại
    💰 Từ 200.000 đồng đến 400.000 đồng
    💰 Từ 400.000 đồng đến 600.000 đồng

  Điều 8: Xử phạt người điều khiển xe máy chuyên dùng vi phạm quy tắc giao thông đường
    💰 Từ 400.000 đồng đến 600.000 đồng
    💰 Từ 600.000 đồng đến 800.000 đồng

  Điều 9: Xử phạt người điều khiển xe đạp, xe đạp máy, người điều khiển xe thô sơ khác vi
    💰 Từ 100.000 đồng đến 200.000 đồng
    💰 Từ 150.000 đồng đến 250.000 đồng

  Điều 10: Xử phạt người đi bộ vi phạm quy tắc giao thông đường bộ
    💰 Từ 150.000 đồng đến 250.000 đồng
    💰 Từ 400.000 đồng đến 600.000 đồng


## 5. Lưu dữ liệu JSON

In [7]:
# Tạo cấu trúc dữ liệu hoàn chỉnh
nd168_data = {
    "title": "Nghị định 168/2024/NĐ-CP - Quy định xử phạt vi phạm hành chính về trật tự, an toàn giao thông đường bộ",
    "document_number": "168/2024/NĐ-CP",
    "document_type": "Nghị định",
    "issued_by": "Chính phủ",
    "issued_date": "2024-12-26",
    "year": 2024,
    "total_pages": len(pages_data),
    "extraction_date": datetime.now().strftime("%Y-%m-%d"),
    "source_file": "ND-168-PDF.pdf",
    "full_text": all_text,
    "pages": pages_data,
    "chapters": chapters,
    "statistics": {
        "total_chapters": len(chapters),
        "total_articles": len(all_articles),
        "total_penalty_articles": len(penalties)
    }
}

# Lưu file JSON raw
import builtins
json_string = json.dumps(nd168_data, ensure_ascii=False, indent=2)

with builtins.open(str(OUTPUT_JSON), 'w', encoding='utf-8') as f:
    f.write(json_string)

print(f"✓ Đã lưu dữ liệu raw: {OUTPUT_JSON}")
print(f"  Kích thước: {len(json_string) / 1024:.1f} KB")

✓ Đã lưu dữ liệu raw: C:\Users\ASUS-PRO\Desktop\Traffic_Law_Chatbot\data\raw\nd168_xu_phat_2024.json
  Kích thước: 768.7 KB


In [8]:
# Lưu file processed - cấu trúc tối ưu cho retrieval chatbot
processed_data = {
    "title": "Mức xử phạt vi phạm giao thông đường bộ",
    "source": "Nghị định 168/2024/NĐ-CP",
    "extraction_date": datetime.now().strftime("%Y-%m-%d"),
    "total_articles": len(all_articles),
    "articles": [],
    "penalties_summary": penalties
}

# Thêm tất cả các điều với nội dung đầy đủ
for article in all_articles:
    processed_data["articles"].append({
        "article_number": article["article_number"],
        "article_title": article["article_title"],
        "chapter": article.get("chapter", ""),
        "section": article.get("section", ""),
        "content": article.get("content", "").strip()
    })

processed_string = json.dumps(processed_data, ensure_ascii=False, indent=2)

with builtins.open(str(OUTPUT_PROCESSED), 'w', encoding='utf-8') as f:
    f.write(processed_string)

print(f"✓ Đã lưu dữ liệu processed: {OUTPUT_PROCESSED}")
print(f"  Kích thước: {len(processed_string) / 1024:.1f} KB")

✓ Đã lưu dữ liệu processed: C:\Users\ASUS-PRO\Desktop\Traffic_Law_Chatbot\data\processed\nd168_penalties.json
  Kích thước: 295.3 KB


## 6. Kiểm tra kết quả

In [9]:
# Tổng kết kết quả
print("=" * 60)
print("TỔNG KẾT CRAWL NGHỊ ĐỊNH 168/2024/NĐ-CP")
print("=" * 60)
print(f"\n📄 Tài liệu: {nd168_data['title']}")
print(f"📝 Số hiệu: {nd168_data['document_number']}")
print(f"🏛️ Cơ quan ban hành: {nd168_data['issued_by']}")
print(f"📅 Ngày ban hành: {nd168_data['issued_date']}")
print(f"📖 Tổng số trang: {nd168_data['total_pages']}")

print(f"\n📊 Thống kê:")
print(f"  - Số chương: {len(chapters)}")
total_sections = sum(len(ch['sections']) for ch in chapters)
print(f"  - Số mục: {total_sections}")
print(f"  - Số điều: {len(all_articles)}")
print(f"  - Điều có mức phạt: {len(penalties)}")
print(f"  - Tổng mức phạt tiền: {sum(len(p['fine_ranges']) for p in penalties)}")
print(f"  - Điều có trừ điểm: {sum(1 for p in penalties if p['demerit_points'])}")

print(f"\n📁 File đã tạo:")
print(f"  - Raw JSON: {OUTPUT_JSON}")
print(f"  - Processed JSON: {OUTPUT_PROCESSED}")

# Chi tiết các chương
print(f"\n📋 Chi tiết cấu trúc:")
for ch in chapters:
    total_arts = len(ch["articles"]) + sum(len(s["articles"]) for s in ch["sections"])
    print(f"\n  Chương {ch['chapter_number']}: {ch['chapter_title']}")
    print(f"    Số điều: {total_arts}")
    if ch["sections"]:
        for sec in ch["sections"]:
            print(f"    Mục {sec['section_number']}: {sec['section_title']} ({len(sec['articles'])} điều)")

print(f"\n✅ Hoàn thành crawl Nghị định 168/2024/NĐ-CP!")

TỔNG KẾT CRAWL NGHỊ ĐỊNH 168/2024/NĐ-CP

📄 Tài liệu: Nghị định 168/2024/NĐ-CP - Quy định xử phạt vi phạm hành chính về trật tự, an toàn giao thông đường bộ
📝 Số hiệu: 168/2024/NĐ-CP
🏛️ Cơ quan ban hành: Chính phủ
📅 Ngày ban hành: 2024-12-26
📖 Tổng số trang: 88

📊 Thống kê:
  - Số chương: 6
  - Số mục: 8
  - Số điều: 303
  - Điều có mức phạt: 52
  - Tổng mức phạt tiền: 187
  - Điều có trừ điểm: 0

📁 File đã tạo:
  - Raw JSON: C:\Users\ASUS-PRO\Desktop\Traffic_Law_Chatbot\data\raw\nd168_xu_phat_2024.json
  - Processed JSON: C:\Users\ASUS-PRO\Desktop\Traffic_Law_Chatbot\data\processed\nd168_penalties.json

📋 Chi tiết cấu trúc:

  Chương I: I
    Số điều: 99
    Mục 1: VI PHẠM QUY TẮC GIAO THÔNG ĐƯỜNG BỘ (11 điều)
    Mục 2: VI PHẠM QUY ĐỊNH VỀ PHƯƠNG TIỆN THAM GIA GIAO THÔNG ĐƯỜNG (6 điều)
    Mục 3: VI PHẠM QUY ĐỊNH VỀ NGƯỜI ĐIỀU KHIỂN PHƯƠNG TIỆN THAM GIA (2 điều)
    Mục 4: VI PHẠM QUY ĐỊNH VỀ BẢO ĐẢM TRẬT TỰ, AN TOÀN GIAO THÔNG (19 điều)
    Mục 5: CÁC VI PHẠM KHÁC LIÊN QUAN ĐẾN TRẬT 